# H&M 2년 가치기저 M2 이식성 screen (K=1)
Dunnhumby에서 M1을 넘고 CLV 순열도 이긴 가치기저 M2가 H&M에서도 개선되는지 봅니다. 원 논문 BPR(음성 1개), seed 42, 2020-09-01까지 학습 → 09-02~08 validation, 100 epoch, batch 131,072입니다.

arm은 **M1 → M2(실제 배정) → M2(순열 배정)** 순서로 학습하고 각 arm 결과를 Drive에 저장합니다. 두 번째 arm까지만 끝나도 '개선되는가'의 답은 나옵니다. 중단되면 같은 노트북을 다시 실행하면 체크포인트에서 이어서 진행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '51176e9bc30a3945a23e1f974cad44b0c07b2f0a'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_m5_value_basis_hm2y_screen as hm_screen

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert hm_screen.CODE_VERSION == 'm5-value-basis-hm2y-k1-screen-v1'
cfg = hm_screen.configure_hm2y_value_basis_screen()
summary = hm_screen.preflight_summary(cfg)
assert summary['dataset'] == 'hm'
assert summary['loss']['negative_count'] == 1
assert summary['m2']['rho'] == 0.25
assert summary['fixed']['batch_size'] == 131072
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# 순열 arm 없이 두 arm만 돌리려면 include_shuffle=False 로 바꾸세요.
result_df = hm_screen.run_hm2y_value_basis_screen(
    hm_screen.configure_hm2y_value_basis_screen(include_shuffle=True)
)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

core = ['model_id', 'm2_assignment', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
        'price_purchase_amount_weighted_hit@10', 'vndcg@10', 'user_value_tendency_recommended_price_alignment']
print('1) 절대지표')
show(result_df[[column for column in core if column in result_df.columns]])
print('2) Top-10 변경 비율 (M1 대비)')
show(result_df.attrs['top10_overlap'])
print('3) 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
